In [5]:
import os
import sys
import django
import pandas as pd
import numpy as np

cwd = os.getcwd()

BASE_DIR = os.path.abspath(os.path.join(cwd, '..', '..'))

if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)
    print(f"Adicionado {BASE_DIR} ao sys.path")

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "gnosis.settings")

django.setup()

from django.db.models import F
from api.models import *
from asgiref.sync import sync_to_async
from authapp.models import User

In [6]:
async def get_marketdata_df():
    qs = await sync_to_async(list)(
        MarketData.objects.values(
            'date', 'close', 'high', 'low', 'open', 'volume',
            symbol=F('asset__symbol'),
        )
    )
    df = pd.DataFrame(qs)
    return df

# Executa e recebe o DataFrame
df_marketdata = await get_marketdata_df()
print(df_marketdata.head())

         date  close   high    low   open  volume symbol
0  1927-12-30  17.66  17.66  17.66  17.66       0  ^GSPC
1  1928-01-03  17.76  17.76  17.76  17.76       0  ^GSPC
2  1928-01-04  17.72  17.72  17.72  17.72       0  ^GSPC
3  1928-01-05  17.55  17.55  17.55  17.55       0  ^GSPC
4  1928-01-06  17.66  17.66  17.66  17.66       0  ^GSPC


In [7]:
df_marketdata.info()  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 274917 entries, 0 to 274916
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   date    274917 non-null  object
 1   close   274917 non-null  object
 2   high    274917 non-null  object
 3   low     274917 non-null  object
 4   open    274917 non-null  object
 5   volume  274917 non-null  int64 
 6   symbol  274917 non-null  object
dtypes: int64(1), object(6)
memory usage: 14.7+ MB


In [8]:
print("Symbols no BD:", df_marketdata['symbol'].unique())

Symbols no BD: ['^GSPC' 'M2SL' '^FVX' 'NFCI' 'SPGI' 'DHR' 'AAPL' 'UNH' 'TEDRATE' 'MSFT'
 '^BVSP' 'BRFS' 'AMZN' 'NVDA' 'PETR4.SA' 'VALE3.SA' 'WEGE3.SA' 'ITSA4.SA'
 'ABEV3.SA' 'DX=F' 'ZC=F' 'CL=F' 'NG=F' 'GC=F' 'ZS=F' 'ITUB4.SA' 'BRL=X'
 'GOOGL' 'Selic_Over' 'Selic_Over_Long' 'CDI' 'Swap_DI_5Y' 'CSAN3.SA'
 'GSG' 'BZ=F' 'B3SA3.SA' 'BBDC4.SA' 'BTGIMABFIRF.SA' 'TSLA' 'META'
 'BTC-USD' 'NTNS11.SA']


In [9]:
aapl_df = df_marketdata[df_marketdata['symbol'] == 'AAPL']


In [10]:
for i,j  in df_marketdata.groupby('symbol').size().items():
    print(f"Symbol: {i}, Registros: {j}")

Symbol: AAPL, Registros: 11395
Symbol: ABEV3.SA, Registros: 6565
Symbol: AMZN, Registros: 7243
Symbol: B3SA3.SA, Registros: 4550
Symbol: BBDC4.SA, Registros: 4511
Symbol: BRFS, Registros: 7223
Symbol: BRL=X, Registros: 5354
Symbol: BTC-USD, Registros: 4184
Symbol: BTGIMABFIRF.SA, Registros: 208
Symbol: BZ=F, Registros: 4624
Symbol: CDI, Registros: 5116
Symbol: CL=F, Registros: 6406
Symbol: CSAN3.SA, Registros: 5044
Symbol: DHR, Registros: 11889
Symbol: DX=F, Registros: 6487
Symbol: GC=F, Registros: 6397
Symbol: GOOGL, Registros: 5417
Symbol: GSG, Registros: 4933
Symbol: ITSA4.SA, Registros: 6567
Symbol: ITUB4.SA, Registros: 6314
Symbol: M2SL, Registros: 805
Symbol: META, Registros: 3465
Symbol: MSFT, Registros: 10069
Symbol: NFCI, Registros: 2877
Symbol: NG=F, Registros: 6403
Symbol: NTNS11.SA, Registros: 504
Symbol: NVDA, Registros: 6818
Symbol: PETR4.SA, Registros: 6567
Symbol: SPGI, Registros: 13369
Symbol: Selic_Over, Registros: 244
Symbol: Selic_Over_Long, Registros: 244
Symbol: S

In [11]:
ptr4 = df_marketdata[df_marketdata['symbol'] == 'ABEV3.SA']

In [12]:
ptr4.tail()

,date,close,high,low,open,volume,symbol
274742,2026-02-24,16.58,16.58,16.23,16.23,31221100,ABEV3.SA
274777,2026-02-25,16.38,16.64,16.26,16.61,29966400,ABEV3.SA
274812,2026-02-26,16.41,16.60,16.36,16.42,18040900,ABEV3.SA
274847,2026-02-27,16.27,16.52,16.25,16.32,34463800,ABEV3.SA
274884,2026-03-02,15.91,16.20,15.87,15.97,18638600,ABEV3.SA


In [13]:
async def get_predictiondata_df():
    qs = await sync_to_async(list)(
        Prediction.objects.values(
            'id', 'date', 'symbol', 'steps_out',
            'mae', 'loss', 'n_features',
            'window_size', 'epochs', 'batch_size', 'scaler_type',
            asset_symbol=F('asset__symbol'),
        )
    )
    return pd.DataFrame(qs)

df_prediction = await get_predictiondata_df()
df_prediction.head()

,id,date,symbol,steps_out,mae,loss,n_features,window_size,epochs,batch_size,scaler_type,asset_symbol
0,65,2025-10-16,AAPL,5,0.032063,0.001659,5,22,60,32,minmax,AAPL
1,66,2025-10-16,ABEV3.SA,5,0.012373,0.000304,5,22,60,32,minmax,ABEV3.SA
2,67,2025-10-16,AMZN,5,0.031452,0.001606,5,22,60,32,minmax,AMZN
3,68,2025-10-16,B3SA3.SA,5,0.016988,0.000509,5,22,60,32,minmax,B3SA3.SA
4,69,2025-10-16,BBDC4.SA,5,0.015609,0.000461,5,22,60,32,minmax,BBDC4.SA


In [14]:
wege3_pred = df_prediction[df_prediction['symbol'] == 'WEGE3.SA']


In [15]:
# Os valores previstos agora vivem na tabela PredictionPoint (um por horizonte/dia).
async def get_prediction_points(symbol):
    return await sync_to_async(list)(
        PredictionPoint.objects
        .filter(prediction__symbol=symbol)
        .values('prediction__date', 'horizon', 'value')
        .order_by('prediction__date', 'horizon')
    )

wege3_points = pd.DataFrame(await get_prediction_points('WEGE3.SA'))
if not wege3_points.empty:
    wege3_series = wege3_points.pivot(index='prediction__date', columns='horizon', values='value')
    print(wege3_series.tail())

horizon                   1          2          3          4          5
prediction__date                                                       
2026-02-27        48.033176  48.001869  47.102432  47.337093  47.057610
2026-02-28        49.583088  48.338070  48.154064  47.842548  47.351803
2026-03-01        48.690891  48.285297  47.780357  47.235752  46.673080
2026-03-02        47.654030  46.415649  46.444031  45.216251  44.662201
2026-03-03        46.531361  46.784592  46.178146  44.817123  45.240978


In [16]:
# As metricas (mae/loss) agora sao colunas escalares do proprio Prediction.
for _, row in df_prediction.head(20).iterrows():
    print(f"{row['symbol']}, {row['date']}: mae={row['mae']}, loss={row['loss']}")

metrics_df = df_prediction[['mae', 'loss']].dropna()

AAPL, 2025-10-16: mae=0.032063182443380356, loss=0.0016586844576522708
ABEV3.SA, 2025-10-16: mae=0.012372687458992004, loss=0.000304333952954039
AMZN, 2025-10-16: mae=0.03145165741443634, loss=0.0016062386566773057
B3SA3.SA, 2025-10-16: mae=0.01698801852762699, loss=0.0005087834433652461
BBDC4.SA, 2025-10-16: mae=0.015609415248036385, loss=0.0004614948411472142
BRFS, 2025-10-16: mae=0.007679832633584738, loss=8.997957047540694e-05
BRL=X, 2025-10-16: mae=0.020095832645893097, loss=0.0006019059219397604
BTC-USD, 2025-10-16: mae=0.04208812117576599, loss=0.0031094427686184645
BZ=F, 2025-10-16: mae=0.0136646069586277, loss=0.0003603784425649792
CDI, 2025-10-16: mae=0.010821052826941013, loss=0.0008818574133329093
CDS_USA_5Y, 2025-10-16: mae=0.017232272773981094, loss=0.0006700585363432765
CL=F, 2025-10-16: mae=0.010097546502947807, loss=0.0001899718918139115
CSAN3.SA, 2025-10-16: mae=0.02454523928463459, loss=0.0008776559843681753
DHR, 2025-10-16: mae=0.023635920137166977, loss=0.001041322

In [17]:
metrics_df.describe()

,mae,loss
count,2915.000000,2915.000000
mean,0.029026,0.002791
std,0.027228,0.009186
min,0.001024,0.000002
25%,0.014650,0.000389
50%,0.020829,0.000748
75%,0.032389,0.001819
max,0.324037,0.213188


In [18]:
# O universo negociavel agora e marcado no catalogo (Asset.is_allowed).
ALLOWED_SYMBOLS = await sync_to_async(list)(
    Asset.objects.filter(is_allowed=True).values_list('symbol', flat=True)
)
print(ALLOWED_SYMBOLS)

['AAPL', 'ABEV3.SA', 'AMZN', 'B3SA3.SA', 'BBDC4.SA', 'BRFS', 'CSAN3.SA', 'DHR', 'GOOGL', 'ITSA4.SA', 'ITUB4.SA', 'META', 'MSFT', 'NVDA', 'PETR4.SA', 'SPGI', 'TSLA', 'UNH', 'VALE3.SA', 'WEGE3.SA']


In [19]:
allowed_symbols_last_predictions = df_prediction[df_prediction['symbol'].isin(ALLOWED_SYMBOLS)].sort_values(by='date').groupby('symbol').tail(1)

In [20]:
# 'used_features' vem da tabela relacional PredictionFeature (ativo + correlacao).
feature_rows = await sync_to_async(list)(
    PredictionFeature.objects.values('prediction_id', 'feature_asset__symbol', 'correlation')
)
features_by_pred = {}
for r in feature_rows:
    features_by_pred.setdefault(r['prediction_id'], []).append(
        [r['feature_asset__symbol'], r['correlation']]
    )

allowed_symbols_last_predictions = allowed_symbols_last_predictions.copy()
allowed_symbols_last_predictions['used_features'] = allowed_symbols_last_predictions['id'].map(features_by_pred)
# mae ja e coluna; 'mse' era o antigo results['metrics']['loss']
allowed_symbols_last_predictions['mse'] = allowed_symbols_last_predictions['loss']

In [21]:
allowed_symbols_last_predictions = allowed_symbols_last_predictions.drop(columns=['id', 'date', 'loss'])

In [22]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

In [23]:
allowed_symbols_last_predictions.to_excel('allowed_symbols_last_predictions.xlsx', index=False)

In [24]:
# Portfolio nao tem mais o JSON 'assets'; mantem apenas as distribuicoes (dicts de pesos).
async def get_portfolios_df():
    qs = await sync_to_async(list)(
        Portfolio.objects.values(
            'id', 'name', 'user_id', 'initial_balance', 'current_balance',
            'initial_distribution', 'current_distribution', 'created_at',
        )
    )
    return pd.DataFrame(qs)

df_portfolios = await get_portfolios_df()
df_portfolios.head()

,id,name,user_id,initial_balance,current_balance,initial_distribution,current_distribution,created_at
0,23,Portfolio Inicial Sugerido,3,4993.00,4993.00,"{'ITUB4.SA': 0.182662, 'PETR4.SA': 0.149464, '...","{'ITUB4.SA': 0.182662, 'PETR4.SA': 0.149464, '...",2026-02-22 19:31:21.099516+00:00
1,22,primeiro,16,318.60,318.60,{'PETR4.SA': 1.0},{'PETR4.SA': 1.0},2025-12-11 17:12:59.265958+00:00
2,21,portifolio novo,12,1172.67,1172.67,"{'MSFT': 0.938031, 'VALE3.SA': 0.061969}","{'MSFT': 0.938031, 'VALE3.SA': 0.061969}",2025-12-11 00:40:38.416117+00:00
3,20,novo portifólio,12,7710.53,7710.53,"{'MSFT': 0.095523, 'VALE3.SA': 0.904477}","{'MSFT': 0.095523, 'VALE3.SA': 0.904477}",2025-12-11 00:26:44.470875+00:00
4,19,Carteira Cheatada,3,1374.80,1374.80,"{'MSFT': 0.35144, 'VALE3.SA': 0.102153, 'AMZN'...","{'MSFT': 0.35144, 'VALE3.SA': 0.102153, 'AMZN'...",2025-12-09 22:10:32.196692+00:00


In [25]:
# Os ativos da carteira agora vivem na tabela relacional PortfolioAsset.
async def get_holdings_df():
    qs = await sync_to_async(list)(
        PortfolioAsset.objects.values(
            'portfolio_id', 'quantity', 'price', symbol=F('asset__symbol'),
        )
    )
    return pd.DataFrame(qs)

df_holdings = await get_holdings_df()
df_holdings.head(20)

,portfolio_id,quantity,price,symbol
0,23,20.44000000,44.62,ITUB4.SA
1,23,19.89000000,37.52,PETR4.SA
2,23,45.67000000,15.30,ABEV3.SA
3,23,1.34000000,414.19,MSFT
4,23,9.10000000,51.40,WEGE3.SA
5,23,28.46000000,13.58,ITSA4.SA
6,23,1.27000000,275.92,UNH
7,23,0.74000000,465.51,SPGI
8,23,0.38000000,668.99,META
9,23,2.11000000,89.43,VALE3.SA


In [30]:
async def get_trackings_df():
    qs = await sync_to_async(list)(
        PortfolioTracking.objects.values('id', 'portfolio_id', 'date', 'balance')
    )
    return pd.DataFrame(qs)

df_trackings = await get_trackings_df()

# A distribuicao de cada tracking agora esta em PortfolioTrackingAsset (peso, quantidade, preco cotado).
async def get_tracking_positions_df():
    qs = await sync_to_async(list)(
        PortfolioTrackingAsset.objects.values(
            'tracking_id', 'weight', 'quantity', 'quoted_price',
            symbol=F('asset__symbol'),
        )
    )
    return pd.DataFrame(qs)

df_tracking_positions = await get_tracking_positions_df()
df_trackings.head()

,id,portfolio_id,date,balance
0,1,9,2025-10-18 00:00:00+00:00,12170.00
1,2,10,2025-10-19 11:19:59.286217+00:00,1930.77
2,11,10,2025-10-21 20:42:57.127456+00:00,1948.83
3,12,9,2025-10-21 20:42:57.146760+00:00,12505.85
4,14,9,2025-10-22 01:15:00.148741+00:00,12505.85


In [27]:
portfolio_10_data =  df_trackings[df_trackings['portfolio_id'] == 10]

In [28]:
portfolio_10_data

,id,portfolio_id,date,balance
1,2,10,2025-10-19 11:19:59.286217+00:00,1930.77
2,11,10,2025-10-21 20:42:57.127456+00:00,1948.83
8,19,10,2025-10-23 20:02:11.926830+00:00,1951.07
11,35,10,2025-10-25 14:56:54.848796+00:00,1984.67
14,38,10,2025-10-29 00:00:00+00:00,2050.02
17,41,10,2025-10-30 00:00:00+00:00,2086.79
20,44,10,2025-11-04 00:00:00+00:00,2066.25
23,47,10,2025-11-05 00:00:00+00:00,2100.66
26,50,10,2025-11-06 00:00:00+00:00,2099.68
29,53,10,2025-11-07 00:00:00+00:00,2068.85


In [31]:
async def get_users_df():
    qs = await sync_to_async(list)(User.objects.all().values())
    return pd.DataFrame(qs) 

df_users = await get_users_df()
df_users.head()

,id,password,last_login,is_superuser,username,first_name,last_name,is_staff,is_active,date_joined,email,name,phone,created_at
0,1,pbkdf2_sha256$1000000$jQ0xUvyUaTCF7zkWjeIScc$b...,None,False,matheus123,,,False,True,2025-10-14 01:02:06.780459+00:00,matheus@ematheus.com,Matheus Oli,+55 11 91234-5678,2025-10-14 01:02:07.780487+00:00
1,2,pbkdf2_sha256$1000000$OD2eaiKfIHxHXJd96Sllji$b...,None,False,eliane12345,,,False,True,2025-10-16 22:51:05.387293+00:00,elianeferra@gmail.com,eliane ferraz,+55 11 91234-5678,2025-10-16 22:51:06.952424+00:00
2,3,pbkdf2_sha256$1000000$A8jhTDgnqyjHSr50pRCNsg$y...,None,False,eusoujabrummmco,,,False,True,2025-10-19 19:10:23.012752+00:00,jabrummmco@gmail.com,Jambrumco Junior,+55 88 7777-7777,2025-10-19 19:10:23.912328+00:00
3,4,pbkdf2_sha256$1000000$Wg12CIMfVVS5b5pXc8FB0u$Y...,None,False,userteste2,,,False,True,2025-12-07 19:29:57.520906+00:00,teste2@gmail.com,Tester junior,+55 99 7777-7777,2025-12-07 19:29:58.451653+00:00
4,5,pbkdf2_sha256$1000000$Z6OHLUCmWfmgATBy1iJn28$k...,None,False,userteste22,,,False,True,2025-12-07 19:54:55.731848+00:00,teste22@gmail.com,Tester junior 2,+55 99 7777-7772,2025-12-07 19:54:56.695572+00:00


## Exemplo completo — novo modelo relacional de Portfolio

O antigo JSON aninhado foi substituído por tabelas relacionais:

- **`Portfolio`** — carteira (mantém `initial_distribution`/`current_distribution` como dicts de pesos).
- **`PortfolioAsset`** — posições atuais (holdings): `asset` (FK), `quantity`, `price`. Helper: `Portfolio.holdings()`.
- **`PortfolioTracking`** — ponto do histórico: `date`, `balance`.
- **`PortfolioTrackingAsset`** — 1 linha por ativo em cada tracking: `asset`, `weight`, `quantity`, `quoted_price`. Helper: `PortfolioTracking.distribution_map()` → `{symbol: peso}`.

A escrita passa pela **camada de serviço** (`PortfolioService`), que mantém todas essas tabelas em sincronia.

In [32]:
from asgiref.sync import sync_to_async

# --- LEITURA: visão geral de uma carteira real, navegando as relações ---
@sync_to_async
def portfolio_overview(portfolio_id):
    p = Portfolio.objects.get(id=portfolio_id)
    last = p.tracking_data.order_by('date').last()
    overview = {
        'id': p.id,
        'name': p.name,
        'user': p.user.email,
        'current_balance': float(p.current_balance),
        'holdings (PortfolioAsset)': p.holdings(),               # [{symbol, quantity, price, asset_id}]
        'n_trackings': p.tracking_data.count(),
    }
    if last:
        overview['last_tracking_date'] = str(last.date)
        overview['last_distribution (distribution_map)'] = last.distribution_map()   # {symbol: peso}
        overview['last_positions (PortfolioTrackingAsset)'] = [
            {
                'symbol': pos.asset.symbol,
                'weight': float(pos.weight),
                'quantity': float(pos.quantity) if pos.quantity is not None else None,
                'quoted_price': float(pos.quoted_price) if pos.quoted_price is not None else None,
            }
            for pos in last.asset_positions.select_related('asset')
        ]
    return overview

@sync_to_async
def pick_portfolio():
    p = (Portfolio.objects
         .filter(asset_holdings__isnull=False, tracking_data__isnull=False)
         .distinct().first())
    return p.id if p else None

pid = await pick_portfolio()
overview = await portfolio_overview(pid)
overview

{'id': 23,
 'name': 'Portfolio Inicial Sugerido',
 'user': 'jabrummmco@gmail.com',
 'current_balance': 4993.0,
 'holdings (PortfolioAsset)': [{'symbol': 'ITUB4.SA',
   'quantity': 20.44,
   'price': 44.62,
   'asset_id': 3},
  {'symbol': 'PETR4.SA', 'quantity': 19.89, 'price': 37.52, 'asset_id': 1},
  {'symbol': 'ABEV3.SA', 'quantity': 45.67, 'price': 15.3, 'asset_id': 5},
  {'symbol': 'MSFT', 'quantity': 1.34, 'price': 414.19, 'asset_id': 13},
  {'symbol': 'WEGE3.SA', 'quantity': 9.1, 'price': 51.4, 'asset_id': 6},
  {'symbol': 'ITSA4.SA', 'quantity': 28.46, 'price': 13.58, 'asset_id': 8},
  {'symbol': 'UNH', 'quantity': 1.27, 'price': 275.92, 'asset_id': 18},
  {'symbol': 'SPGI', 'quantity': 0.74, 'price': 465.51, 'asset_id': 20},
  {'symbol': 'META', 'quantity': 0.38, 'price': 668.99, 'asset_id': 16},
  {'symbol': 'VALE3.SA', 'quantity': 2.11, 'price': 89.43, 'asset_id': 2},
  {'symbol': 'NVDA', 'quantity': 0.3, 'price': 174.19, 'asset_id': 12},
  {'symbol': 'GOOGL', 'quantity': 0.1

In [33]:
# --- Navegando o grafo de FKs: Portfolio -> PortfolioAsset -> Asset -> AssetType ---
@sync_to_async
def holdings_table(portfolio_id):
    p = Portfolio.objects.get(id=portfolio_id)
    rows = [
        {
            'symbol': h.asset.symbol,
            'tipo': h.asset.asset_type.name,
            'is_allowed': h.asset.is_allowed,
            'quantity': float(h.quantity),
            'price': float(h.price),
            'valor': float(h.quantity) * float(h.price),
        }
        for h in p.asset_holdings.select_related('asset', 'asset__asset_type')
    ]
    return pd.DataFrame(rows)

await holdings_table(pid)

,symbol,tipo,is_allowed,quantity,price,valor
0,ITUB4.SA,stock,True,20.44,44.62,912.0328
1,PETR4.SA,stock,True,19.89,37.52,746.2728
2,ABEV3.SA,stock,True,45.67,15.30,698.7510
3,MSFT,stock,True,1.34,414.19,555.0146
4,WEGE3.SA,stock,True,9.10,51.40,467.7400
5,ITSA4.SA,stock,True,28.46,13.58,386.4868
6,UNH,stock,True,1.27,275.92,350.4184
7,SPGI,stock,True,0.74,465.51,344.4774
8,META,stock,True,0.38,668.99,254.2162
9,VALE3.SA,stock,True,2.11,89.43,188.6973


### Criar e atualizar uma carteira pela camada de serviço

Tudo em uma transação com **rollback** — demonstra o caminho de escrita sem persistir nada no banco.

In [34]:
from django.db import transaction
from api.portfolios.services import PortfolioService

@sync_to_async
def portfolio_write_demo():
    from authapp.models import User

    class _Rollback(Exception):
        pass

    result = {}
    try:
        with transaction.atomic():
            user = User.objects.first()
            assets = [
                {'symbol': 'AAPL',     'quantity': 10, 'price': 180.0},
                {'symbol': 'MSFT',     'quantity': 5,  'price': 410.0},
                {'symbol': 'PETR4.SA', 'quantity': 100, 'price': 38.0},
            ]

            # 1) regra de negócio: distribuição de pesos (soma 1.0, com correção de arredondamento)
            total, dist = PortfolioService.compute_distribution(assets, strict=True, skip_non_positive=True)
            result['1_distribuicao'] = {'total': float(total), 'pesos': dist}

            # 2) cria a carteira -> grava Portfolio + PortfolioAsset + PortfolioTracking + PortfolioTrackingAsset
            p = PortfolioService.create_portfolio(user, 'Exemplo Gnosis', 'demo', assets, dist, total)
            t = p.tracking_data.first()
            result['2_criada'] = {
                'id': p.id,
                'saldo': float(p.current_balance),
                'holdings': p.holdings(),
                'tracking_inicial': {'date': str(t.date), 'balance': float(t.balance),
                                     'distribution_map': t.distribution_map()},
            }

            # 3) adiciona um ativo -> recalcula pesos e gera um novo ponto de histórico
            novos = p.holdings() + [{'symbol': 'NVDA', 'quantity': 3, 'price': 170.0}]
            PortfolioService.apply_assets_update(p, novos)
            result['3_apos_add_NVDA'] = {
                'holdings': [h['symbol'] for h in p.holdings()],
                'n_trackings': p.tracking_data.count(),
                'nova_distribuicao': p.current_distribution,
            }

            raise _Rollback()   # desfaz tudo — nada é persistido
    except _Rollback:
        pass
    return result

await portfolio_write_demo()

c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\django\db\models\fields\__init__.py:1671: RuntimeWarning: DateTimeField PortfolioTracking.date received a naive datetime (2026-08-01 10:55:24.588900) while time zone support is active.
  warnings.warn(
c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\django\db\models\fields\__init__.py:1671: RuntimeWarning: DateTimeField PortfolioTracking.date received a naive datetime (2026-08-01 10:55:24.604526) while time zone support is active.
  warnings.warn(


{'1_distribuicao': {'total': 7650.0,
  'pesos': {'AAPL': 0.235294, 'MSFT': 0.267974, 'PETR4.SA': 0.496732}},
 '2_criada': {'id': 24,
  'saldo': 7650.0,
  'holdings': [{'symbol': 'AAPL',
    'quantity': 10.0,
    'price': 180.0,
    'asset_id': 11},
   {'symbol': 'MSFT', 'quantity': 5.0, 'price': 410.0, 'asset_id': 13},
   {'symbol': 'PETR4.SA', 'quantity': 100.0, 'price': 38.0, 'asset_id': 1}],
  'tracking_inicial': {'date': '2026-08-01 10:55:24.588900+00:00',
   'balance': 7650.0,
   'distribution_map': {'AAPL': 0.235294,
    'MSFT': 0.267974,
    'PETR4.SA': 0.496732}}},
 '3_apos_add_NVDA': {'holdings': ['AAPL', 'MSFT', 'PETR4.SA', 'NVDA'],
  'n_trackings': 2,
  'nova_distribuicao': {'AAPL': 0.220588,
   'MSFT': 0.251225,
   'PETR4.SA': 0.465686,
   'NVDA': 0.06250100000000003}}}

### PnL histórico da carteira

Usa `PortfolioPnlCalculator` (em `portfolios/src`), que lê o histórico via `distribution_map()` e os preços de `MarketData` (por `asset__symbol`).

In [35]:
@sync_to_async
def portfolio_pnl(portfolio_id):
    from api.portfolios.src.pnl_measurements import PortfolioPnlCalculator

    p = Portfolio.objects.get(id=portfolio_id)
    tracking = PortfolioService.tracking(p)
    symbols = set(tracking.first().distribution_map()) | set(tracking.last().distribution_map())
    market = MarketData.objects.filter(asset__symbol__in=symbols).order_by('date')
    return PortfolioPnlCalculator(tracking_records=tracking, symbols_data=market, assets=p.holdings()).calculate()

pnl = await portfolio_pnl(pid)
print('Resumo:', pnl['pnl_general'])
pd.DataFrame(pnl['pnl_data'])

Resumo: {'initial_balance': 4993.0, 'current_balance': 4963.06, 'total_pnl_value': -29.94, 'total_pnl_percent': -0.6, 'initial_date': '2026-02-22T16:31:21.124811+00:00', 'current_date': '2026-03-02T00:00:00+00:00', 'balance_volatility': np.float64(0.1455)}


,symbol,quantity,average_price,current_price,week_ago_price,initial_value,current_value,pnl_value,pnl_percent,week_change_percent,initial_weight,current_weight
0,VALE3.SA,2.11,89.43,88.16,88.16,188.70,191.63,2.93,1.56,0.0,3.78,3.86
1,WEGE3.SA,9.10,51.40,48.69,48.69,467.74,438.05,-29.69,-6.35,0.0,9.37,8.83
2,META,0.38,669.00,653.56,653.56,254.22,253.40,-0.81,-0.32,0.0,5.09,5.11
3,UNH,1.27,275.92,294.93,294.93,350.42,356.38,5.96,1.70,0.0,7.02,7.18
4,ABEV3.SA,45.67,15.30,15.91,15.91,698.75,693.52,-5.23,-0.75,0.0,13.99,13.97
5,SPGI,0.74,465.51,443.08,443.08,344.48,365.60,21.12,6.13,0.0,6.90,7.37
6,GOOGL,0.11,333.03,306.52,306.52,36.63,35.65,-0.98,-2.69,0.0,0.73,0.72
7,MSFT,1.34,414.19,398.55,398.55,555.02,556.86,1.84,0.33,0.0,11.12,11.22
8,ITUB4.SA,20.44,44.62,45.92,45.92,912.03,850.71,-61.32,-6.72,0.0,18.27,17.14
9,ITSA4.SA,28.46,13.58,14.14,14.14,386.49,362.64,-23.85,-6.17,0.0,7.74,7.31
